# KAG: Knowledge-Augmented Generation (graph-based)

**Idea:** store the knowledge as a **graph** of facts: *subject → relation → object*.
Find the entities named in the question, then **follow the links** hop by hop to collect
facts that are connected, even when no single document states them together.

```
question ─► find entities ─► walk the graph (hop 1, hop 2 ...) ─► connected facts + Claude ─► answer
```

Here the graph is written by hand in `data/graph.csv`. Real KAG (for example OpenSPG KAG)
uses an LLM to **extract** the graph from the documents and to **plan** the hops.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent            # the demos/ folder
DATA = ROOT / "data"                # the knowledge: the same files for all three demos
assert DATA.exists(), "Open this notebook from demos/notebooks/"
sys.path.append(str(ROOT))
import importlib, llm
importlib.reload(llm)               # picks up edits to llm.py when this cell is re-run
from llm import ask                 # small helper that calls Claude and shows the chat

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque

## 1. The knowledge graph: one row per fact

In [ ]:
facts = pd.read_csv(DATA / "graph.csv")
facts

## 2. Build and draw the graph

In [ ]:
G = nx.DiGraph()
for s, r, o in facts.itertuples(index=False):
    G.add_edge(s, o, relation=r)
print(G.number_of_nodes(), "entities,", G.number_of_edges(), "facts")

# ---- Everything below is only for drawing; the walk in step 4 uses G alone ----------------
from matplotlib.patches import FancyArrowPatch, Patch

COL_GAP = 2.6  # horizontal space between columns

KINDS = {  # kind of entity: (column, colour, legend label)
    "person":  (0, "#cfe3ff", "person"),
    "team":    (1, "#c6efe0", "team"),
    "manager": (2, "#e2d6ff", "team lead"),
    "office":  (2, "#ececec", "office"),
    "role":    (3, "#ffe0b0", "role"),
    "rule":    (4, "#ffd0d6", "policy rule"),
    "value":   (5, "#f4f4f4", "value"),
}

def kind_of(n):
    out = {d["relation"] for *_, d in G.out_edges(n, data=True)}
    inn = {d["relation"] for *_, d in G.in_edges(n, data=True)}
    if "member of" in out:                             return "person"
    if "member of" in inn or "line manager is" in out: return "team"
    if "line manager is" in inn:                       return "manager"
    if "based in" in inn:                              return "office"
    if "has role" in inn:                              return "role"
    return "rule" if out else "value"

kind = {n: kind_of(n) for n in G}

# Place each entity level with the entities it links to, so links don't cross.
pos = {}
for col in range(6):
    nodes = [n for n in G if KINDS[kind[n]][0] == col]
    want = {}
    for n in nodes:
        placed = [pos[m][1] for m in nx.all_neighbors(G, n) if m in pos]
        want[n] = sum(placed) / len(placed) if placed else None
    order = sorted(nodes, key=lambda n: (want[n] is None, want[n] or 0, nodes.index(n)))
    ys = []
    for n in order:
        target = want[n] if want[n] is not None else (ys[-1] + 1 if ys else 0)
        ys.append(max(target, ys[-1] + 1) if ys else target)       # at least 1 row apart
    known = [(want[n], y) for n, y in zip(order, ys) if want[n] is not None]
    shift = sum(w - y for w, y in known) / len(known) if known else 0
    for n, y in zip(order, ys):
        pos[n] = (col * COL_GAP, y + shift)

def draw(highlight=(), start=(), current_hop=None, chain=(), title=None, show=True):
    """highlight: walked facts (subject, relation, object, hop).  current_hop: the hop just taken (drawn
    brighter, earlier hops paler).  chain: entities on the answer path (everything else fades)."""
    walked = {(s, o): hop for s, _, o, hop in highlight}
    links = set(zip(chain, chain[1:]))
    chain_edges = links | {(b, a) for a, b in links}
    reached = set(chain) if chain else {n for e in walked for n in e} | set(start)
    focus = bool(highlight or start)
    fig, ax = plt.subplots(figsize=(11.5, 6.8), dpi=85)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.93, bottom=0.07)
    boxes = {}
    for n, (x, y) in pos.items():
        faded = focus and n not in reached
        boxes[n] = ax.text(x, y, n.replace(" ", "\n", 1), ha="center", va="center", fontsize=9.5, zorder=3,
                           color="#999999" if faded else "#1a1a1a", weight="bold" if n in start else "normal",
                           bbox=dict(boxstyle="round,pad=0.45", fc="#f6f6f6" if faded else KINDS[kind[n]][1],
                                     ec="#d4a017" if n in start else ("#dddddd" if faded else "#9a9a9a"),
                                     lw=2.5 if n in start else 0.8))
    for s, o, d in G.edges(data=True):
        hop = walked.get((s, o))
        if chain:
            on = strong = (s, o) in chain_edges
        else:
            on = hop is not None
            strong = on and current_hop in (None, hop)
        colour = "#1b8a3a" if strong else ("#9fd3ae" if on else ("#d0d0d0" if focus else "#7a7a7a"))
        ax.add_patch(FancyArrowPatch(pos[s], pos[o], patchA=boxes[s].get_bbox_patch(), patchB=boxes[o].get_bbox_patch(),
                                     arrowstyle="-|>", mutation_scale=15, lw=2.6 if strong else (1.6 if on else 1),
                                     color=colour, zorder=3.5, shrinkA=1, shrinkB=1))
        (x1, y1), (x2, y2) = pos[s], pos[o]
        label = f"{d['relation']}\nhop {hop}" if on and hop else d["relation"]
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, label, ha="center", va="center", fontsize=7.5, zorder=5,
                color=colour if (on or not focus) else "#b5b5b5", weight="bold" if strong else "normal",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9))
    ys = [y for _, y in pos.values()]
    ax.set_xlim(-0.9, 5 * COL_GAP + 0.9); ax.set_ylim(max(ys) + 0.8, min(ys) - 0.8); ax.axis("off")
    fig.legend(handles=[Patch(fc=c, ec="#9a9a9a", label=l) for _, c, l in KINDS.values()],
               loc="lower center", ncol=7, frameon=False, fontsize=8)
    if title:
        ax.set_title(title, fontsize=11.5, loc="left", weight="bold")
    elif focus:
        ax.set_title("Gold border = entities found in the question.  Green = links the walk followed.", fontsize=9)
    if not show:
        return fig
    plt.show()

draw()

## 3. Link the question to the graph

Find the graph nodes the question mentions (a first name or a key word is enough).

In [ ]:
question = "Who approves Omar's leave carry-over?"
q = question.lower()
start = [n for n in G if any(len(w) > 3 and w in q for w in n.lower().split())]
start

## 4. Walk the graph

Breadth-first: first every fact touching the start entities (hop 1), then every fact
touching what we just reached (hop 2), and so on, up to `MAX_HOPS`.

In [ ]:
MAX_HOPS = 2

def walk(start, max_hops=MAX_HOPS):
    found, seen, queue = [], set(start), deque((n, 0) for n in start)
    while queue:
        node, hops = queue.popleft()
        if hops == max_hops:
            continue
        links = [(node, nxt) for nxt in G.successors(node)] + [(prev, node) for prev in G.predecessors(node)]
        for s, o in links:                                # follow links in both directions
            fact = (s, G.edges[s, o]["relation"], o)
            if fact not in [f[:3] for f in found]:
                found.append((*fact, hops + 1))
            nxt = o if s == node else s
            if nxt not in seen:
                seen.add(nxt); queue.append((nxt, hops + 1))
    return found

path = walk(start)
pd.DataFrame(path, columns=["subject", "relation", "object", "hop"])

Notice where the walk from **Omar** and the walk from **Leave carry-over** meet: **Sara Ahmed**.

In [ ]:
draw(path, start)

**Watch the walk, one step at a time.** The animation loops, and each step holds for a few seconds.

In [ ]:
import base64, io
from IPython.display import HTML, display
from PIL import Image

def answer_chain(start, path):
    """The chain of walked links that connects two of the entities found in the question."""
    W = nx.Graph([(s, o) for s, _, o, _ in path])
    best = []
    for i, a in enumerate(start):
        for b in start[i + 1:]:
            if a in W and b in W and nx.has_path(W, a, b):
                best = max(best, nx.shortest_path(W, a, b), key=len)
    return best

def chain_text(chain):
    steps = [("→ " if G.has_edge(a, b) else "← ") + b for a, b in zip(chain, chain[1:])]
    return "  ".join([chain[0]] + steps)

def animate_walk(question, start, path):
    hops = sorted({h for *_, h in path})
    chain = answer_chain(start, path)
    frames = [({}, f"Question: {question}", 2500),
              ({"start": start}, "1 · Find the entities the question names (gold)", 3000)]
    for h in hops:
        source = "the gold entities" if h == 1 else f"what hop {h - 1} reached"
        frames.append(({"highlight": [f for f in path if f[3] <= h], "start": start, "current_hop": h},
                       f"{h + 1} · Hop {h}: follow every link from {source}", 3500))
    if chain:
        frames.append(({"highlight": path, "start": [n for n in start if n in chain], "chain": chain},
                       f"{len(hops) + 2} · The walks connect:  " + chain_text(chain), 6500))
    images, durations = [], []
    for kwargs, title, ms in frames:
        fig = draw(title=title, show=False, **kwargs)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=85)
        plt.close(fig)
        images.append(Image.open(buf).convert("RGB"))
        durations.append(ms)
    gif = io.BytesIO()
    images[0].save(gif, format="GIF", save_all=True, append_images=images[1:], duration=durations, loop=0)
    display(HTML(f'<img src="data:image/gif;base64,{base64.b64encode(gif.getvalue()).decode()}" '
                 f'style="max-width:100%">'))

animate_walk(question, start, path)

## 5. Augment and generate

Give Claude the connected facts and ask it to chain them.

In [ ]:
def kag_answer(question, facts):
    context = "\n".join(f"- {s} | {r} | {o}" for s, r, o, *_ in facts)
    system = ("Answer using only these facts from the company knowledge graph. "
              "Chain the facts together and show the path you followed.\n\nFacts:\n" + context)
    return ask(system, question)

kag_answer(question, path);

## Try another person

In [ ]:
question = "Who approves Lina's leave carry-over?"
q = question.lower()
start = [n for n in G if any(len(w) > 3 and w in q for w in n.lower().split())]
path = walk(start)
animate_walk(question, start, path)
kag_answer(question, path);

## Takeaways

- ✅ Answers questions that need **several connected facts** (RAG retrieved Omar's team but missed Sara).
- ✅ Explainable: the answer comes with the path of facts it used.
- ❌ Someone (or an LLM) has to **build and maintain the graph**, which is costly and can introduce errors.
- ❌ Facts that were not extracted into the graph cannot be found.